In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

In [2]:
df=pd.read_csv("train.txt",sep=";",header=None,names=["text","emotion"])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [5]:
unique_emotions=df["emotion"].unique()
emotion_numbers={}
i=0
for emotion in unique_emotions:
  emotion_numbers[emotion]=i
  i+=1
df["emotion"]=df["emotion"].map(emotion_numbers)

In [6]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [7]:
# lowercase
df["text"]=df["text"].apply(lambda x:x.lower())

In [8]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [9]:
# remove punctuations
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))
df["text"]=df["text"].apply(remove_punc)

In [10]:
# remove numbers form text
def remove_num(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new
df["text"]=df["text"].apply(remove_num)


In [11]:
# remove emojis (they only have unicode and doesnot ascii value)
def remove_emoji(txt):
  new=""
  for i in txt:
    if i.isascii():
      new+=i
  return new
df["text"]=df["text"].apply(remove_emoji)

In [12]:
# remove stopwords
import nltk

In [13]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [14]:
nltk.download('punkt') # used for tokenization
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Bhawn\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bhawn\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Bhawn\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
stopwords=set(stopwords.words('english'))
stopwords

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [17]:
def remove_stopwords(txt):
  words=word_tokenize(txt)                          #words=txt.split() 
  cleaned=[]
  for i in words:
    if i not in stopwords:
      cleaned.append(i)
  return ' '.join(cleaned)
df["text"]=df["text"].apply(remove_stopwords)

In [18]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train,X_test,y_train,y_test=train_test_split(df["text"],df["emotion"],test_size=0.2,random_state=42)

In [21]:
X_train.shape

(12800,)

In [22]:
X_test.shape

(3200,)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [24]:
bow_vectorizer=CountVectorizer()

In [25]:
X_train_bow=bow_vectorizer.fit_transform(X_train)
X_test_bow=bow_vectorizer.transform(X_test)

In [26]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [27]:
nb_model=MultinomialNB()

In [28]:
nb_model.fit(X_train_bow,y_train)

MultinomialNB()

In [29]:
pred_nb=nb_model.predict(X_test_bow)

In [30]:
print(accuracy_score(y_test,pred_nb))

0.7678125


In [31]:
tf_idf_model=TfidfVectorizer()

In [32]:
X_train_tfidf=tf_idf_model.fit_transform(X_train)
X_test_tfidf=tf_idf_model.transform(X_test)

In [33]:
nb2_model=MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [34]:
pred_nb2=nb2_model.predict(X_test_tfidf)

In [35]:
accuracy_score(y_test,pred_nb2)

0.6609375

In [36]:
from sklearn.linear_model import LogisticRegression

In [37]:
lr_model=LogisticRegression(max_iter=1000)

In [38]:
lr_model.fit(X_train_bow,y_train)
pred_lr=lr_model.predict(X_test_bow)

In [39]:
accuracy_score(y_test,pred_lr)

0.88875

In [40]:
# saving bow_vectorizer to use it during predictions
pickle.dump(bow_vectorizer,open("bow_vectorizer","wb"))

In [41]:
# lr_model has the highest accuracy score
pickle.dump(lr_model,open("lr_model.pkl","wb"))

In [42]:
lr2_model=LogisticRegression()

In [43]:
lr2_model.fit(X_train_tfidf,y_train)
pred_lr2=lr2_model.predict(X_test_tfidf)

In [44]:
accuracy_score(y_test,pred_lr2)

0.8615625

In [45]:
from sklearn.svm import SVC

In [46]:
svm_model=SVC(kernel="linear")

In [47]:
# SVM - BoW
svm_model.fit(X_train_bow,y_train)
pred_svm=svm_model.predict(X_test_bow)

In [48]:
accuracy_score(y_test,pred_svm)

0.87375

In [49]:
# SVM - TF-IDF
svm2_model=SVC(kernel="linear")
svm2_model.fit(X_train_tfidf,y_train)
pred_svm2=svm2_model.predict(X_test_tfidf)

In [50]:
accuracy_score(y_test,pred_svm2) # 85% without linear

0.883125

In [51]:
import os
print(os.getcwd())

C:\Users\Bhawn\EmotionDetectionUsingMLandNLP
